In [2]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict
import subprocess
import os
import base64

from dotenv import load_dotenv

# .env 파일을 자동으로 환경변수로 로드합니다.
# 이 호출이 없으면 os.environ["GOOGLE_API_KEY"]에서 KeyError가 날 수 있습니다.
load_dotenv()

# LangChain의 Vertex/Gemini 래퍼입니다.
# 현재 노트북에서는 transcribe_audio()가 google-genai(저수준)로 전사하므로
# llm 변수는 “예시 설정” 역할로 남아 있습니다.
from langchain_google_vertexai import ChatVertexAI

# google-genai(= Vertex/Gemini 쪽 공식 클라이언트)입니다.
from google import genai

# ----------------------------
# (참고) LangChain LLM 설정
# ----------------------------
llm = ChatVertexAI(
  model_name="gemini-2.5-flash-lite",
  project="ai-prompt-evaluator-489612",
  location="us-central1",
  max_tokens=500
)

# ----------------------------
# LangGraph에서 사용할 State 정의
# ----------------------------
# State는 “그래프가 다음 노드로 넘어갈 때 들고 다니는 데이터”라고 생각하면 됩니다.
class State(TypedDict):
  # 입력: 다운로드/처리된 원본 영상 파일 경로
  video_file: str

  # 중간 산출물: ffmpeg으로 추출한 오디오 파일 경로
  audio_file: str

  # 출력: 전사(Transcription) 결과 텍스트
  transcription: str
  

C:\Users\m1nk1\AppData\Local\Temp\ipykernel_20668\1565888468.py:24: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(


In [ ]:
def extract_audio(state: State):
  """영상(mp4)에서 오디오(mp3)를 추출합니다."""

  # 예시로 mp4 -> mp3만 단순 치환합니다.
  # (영상이 mp4가 아닐 경우 이 로직은 수정이 필요합니다.)
  output_file = state["video_file"].replace("mp4", "mp3")

  # ffmpeg으로 오디오 추출.
  # -filter:a atempo=2.0 : 오디오 재생 속도를 2배로(선택사항).
  # 전사 정확도에 영향을 줄 수 있으니 필요하면 제거하거나 값 조정하세요.
  command = [
    "ffmpeg",
    "-i",
    state["video_file"],
    "-filter:a",
    "atempo=2.0",
    "-y",  # 같은 파일명이 있으면 덮어쓰기
    output_file,
  ]

  # check=True를 주면 ffmpeg이 실패했을 때 조용히 무시되지 않습니다.
  subprocess.run(command, check=True)

  return {
    "audio_file": output_file,
  }


def transcribe_audio(state: State):
  """오디오(mp3)를 Gemini(google-genai)로 전사해서 텍스트를 반환합니다."""

  # .env 또는 환경변수에 GOOGLE_API_KEY가 있어야 합니다.
  # 이미 프로젝트에 .env가 있으니, 실행 전에 값을 확인하세요.
  api_key = os.environ.get("GOOGLE_API_KEY")
  if not api_key:
    raise ValueError(
      "GOOGLE_API_KEY가 없습니다. .env를 채우고 load_dotenv() 호출이 되었는지 확인하세요."
    )

  client = genai.Client(api_key=api_key)

  # 오디오 파일을 바이너리로 읽어서 base64로 변환합니다.
  with open(state["audio_file"], "rb") as audio_file:
    audio_bytes = audio_file.read()

  audio_b64 = base64.b64encode(audio_bytes).decode("utf-8")

  # 전사 지시문입니다.
  prompt = (
    "Transcribe the audio to Korean. "
    "This country is the Republic of Korea "
    "Return only the transcription text."
  )

  # 텍스트 + 오디오를 한 번에 보내는 멀티모달 입력입니다.
  response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=[
      {
        "role": "user",
        "parts": [
          {"text": prompt},
          {
            "inline_data": {
              # .mp3 파일에 대한 mime type
              "mime_type": "audio/mpeg",
              "data": audio_b64,
            }
          },
        ],
      }
    ],
  )

  # 응답 구조는 버전에 따라 달라질 수 있어 방어적으로 읽습니다.
  transcription = getattr(response, "text", None)
  if not transcription:
    try:
      transcription = response.candidates[0].content.parts[0].text
    except Exception:
      transcription = None

  # 최후의 안전장치
  if not transcription:
    transcription = str(response)

  return {
    "transcription": transcription,
  }
    

In [4]:
# ----------------------------
# LangGraph 그래프 구성
# ----------------------------
# 노드 이름은 add_node(...)에서 지정한 값과
# add_edge(...)에서 참조하는 값이 반드시 같아야 합니다.

graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
# (오타) transcripbe_audio -> transcribe_audio 로 맞춰야 정상 연결됩니다.
graph_builder.add_node("transcribe_audio", transcribe_audio)

# START에서 바로 실행할 노드를 연결합니다.
graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_edge("transcribe_audio", END)

graph = graph_builder.compile()


In [5]:
# ----------------------------
# 그래프 실행(예시)
# ----------------------------
# 실행 흐름:
# 1) extract_audio: ffmpeg으로 korea_2.mp4 -> korea_2.mp3 생성
# 2) transcribe_audio: Gemini로 오디오 전사
#
# 주의:
# - ffmpeg이 설치되어 있고 PATH에 잡혀 있어야 합니다.
# - korea_2.mp4파일이 이 노트북이 실행되는 작업 디렉터리에 있어야 합니다.

graph.invoke({
  "video_file": "korea_2.mp4"
})


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}